In [93]:
# 该模块实现的功能与PyTorch相同，即简化版但拥有核心功能的PyTorch，便于理解可微分编程是怎么运作的，前向和反向传播是什么
import numpy as np
import random
class Value:#定义值类   # tensor

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data                    #数据
        self.grad = 0                       #对结果的影响
        self._backward = lambda: None       #反向传播
        self._prev = set(_children)         #前向传播中影响它的子节点
        self._op = _op                      #操作类型
        self.label = label                  #标签

    def __repr__(self):#输出时包装                     
        return f"Value(data=({self.data}))"

    def __add__(self, other):#重定义+
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():#反向传播算法
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
            
        out._backward = _backward
        return out

    def __pow__(self, other):#need update
        assert isinstance(other, (int, float)), "目前仅支持整型和浮点型变量"
        out = Value(self.data**other, (self, ), f'**{other}')

        def _backward():
            self.grad += (other * self.data**(other - 1)) * out.grad
        
        out._backward = _backward
        return out

    def __truediv__(self, other):
        return self * other**-1

    def __rmul__(self, other):#equal to other * self
        return self * other

    def __radd__(self, other):#equal to other + self
        return self + other

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        
        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad += out.data * out.grad
            
        out._backward = _backward()
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return (-self) + other

    def backward(self):#反向传播操作

        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


In [122]:
class Neuron:#定义神经元

    def __init__(self, nin):#nin是神经元接收到的输入数量
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]#为每个输入创建随机权重
        self.b = Value(random.uniform(-1,1))#创建该神经元的偏置，用于控制该神经元的整体触发频率

    def __call__(self, x):#调用该神经元需要给予数组x
        #希望进行前向传播
        act = sum(wi*xi for wi, xi in zip(self.w, x)) + self.b #进行计算
        out = act.tanh() #激活
        return out

    def parameters(self):#返回参数列表
        return self.w + [self.b]


class Layer:#定义神经元层

    def __init__(self, nin, nout):#nin是单个神经元的输入数量，nout是神经元数量及输出数量
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):#调用神经元层需要输入数组x
        outs = [n(x) for n in self.neurons]#初始化每一个神经元
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):#获取该层每一个神经元的参数
        params = []
        for neuron in self.neurons:
            ps = neuron.parameters()
            params.extend(ps)
        return params

class MLP:#定义多层感知机

    def __init__(self, nin, nouts):#nin是单个神经元的输入数量，nouts为一个数组，定义了MLP中所有层的大小
        sz = [nin] + nouts#所有层的大小，包括输入层
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]#初始化层

    def __call__(self, x):#调用多层感知机需要一个数组x
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):#获取感知机中每一层的参数列表
        params = []
        for layer in self.layers:
            ps = layer.parameters()
            params.extend(ps)
        return params



In [123]:
x = [2.0, 3.0, -1.0]#样例输入
n = MLP(3, [4, 4, 1])#3个输入到含有四个神经元的两层，以及一个输出层
n(x)

Value(data=(-0.3962946108037805))

In [160]:
n = MLP(3, [4, 4, 1])
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]#4组数据
ys = [1.0, -1.0, -1.0, 1.0]#希望得到的结果


In [161]:
ypred = [n(x) for x in xs]
print(ypred)
stsp = 20#循环数
strate = 0.01#学习率
flag = False
for k in range(stsp):#学习循环
    #前向传播
    ypred = [n(x) for x in xs]
    #计算损失
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
    if not flag:
        print(loss)
        flag = True
    #反向传播
    loss.backward()
    #更新数值
    for p in n.parameters():
        p.data += -strate * p.grad  
    #监测
    print(k+1, loss.data)

[Value(data=(0.05611368115657535)), Value(data=(0.9736728639758118)), Value(data=(0.7766356971796095)), Value(data=(0.3432003480800956))]
Value(data=(8.37412614014926))
1 8.37412614014926
2 8.104902485808696
3 7.695333653966521
4 7.303337396689066
5 7.003838357350596
6 6.7772470869571
7 6.554895365779353
8 6.2426982088050735
9 5.704797723458153
10 4.719020801849077
11 3.1098599462087178
12 1.6805543299469736
13 1.2198501424321706
14 0.9776341998204607
15 0.7306188306091281
16 0.49950406245745205
17 0.2945394699977506
18 0.14663426790942624
19 0.06332852379611556
20 0.024899214705863496


In [150]:
ypred

[Value(data=(0.9974366329053779)),
 Value(data=(-0.9999900011737838)),
 Value(data=(-0.9999898540729883)),
 Value(data=(0.9986834036400479))]